In [1]:
# This file is the main file for the WSPI PJK project

#=========== Imports =============#
# To apply math operations on arrays
import numpy as np

# To open and read image files
from PIL import Image

# To get the total amount of files in a folder
from collections import Counter

# Core ML library for building and training neural networks
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# Tools built on torch specifically for image based ML tasks
import torchvision
import torchvision.transforms as transforms
from torchvision import datasets # To load and label images based on folder structure

# To check file names
import os

# For determining time per epoch
import time

import torchvision.transforms.functional as TF # To allow cropping and more image transformations

from torch.utils.data import random_split, DataLoader # To split up teh data and handle batching, shuffling, paralel loading, ect

print("All libraries imported")

All libraries imported


In [2]:
# Check if being run on Google Colab, Kaggle, or locally (Change paths as needed)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    base_path = '/content/drive/Shareddrives/WSPI/catergorized2'
    save_path = '/content/drive/MyDrive/trained_net.pth'
except Exception:
    if os.path.exists('/kaggle'):
        base_path = '/kaggle/input/datasets/johnson183/catergorized2/catergorized2'  # Replace with your actual dataset name
        save_path = '/kaggle/working/trained_net.pth'
    else:
        base_path = './data/images'
        save_path = './trained_net.pth'

#=== Constants ====#
image_size = 224

# Check if the current device has a NAVIDIA GPU to run things on, and use cpu if not
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using: {device}")

Using: cuda


In [3]:
#====== Use ONLY if on google colab to download images to the enviorment (Faster) =========#
# try:
#     import shutil
#     local_data = '/content/data'
#     if not os.path.exists(local_data):
#         print("Copying data to local disk...")
#         shutil.copytree(base_path, local_data)
#         print("Done!")
#     base_path = local_data
# except Exception:
#     pass

In [4]:
# Define how each image will be loaded and preprocessed before given to the model
class RoadDataset(datasets.ImageFolder):

    # Called whenever an image is accessed (ex. data[0])
    def __getitem__(self, index):
        # 1) Get the path, label, image size, and the image itself
        path, label_idx = self.samples[index]
        image = self.loader(path) # Loads as PIL (Python Imaging Library) Image
        image = image.convert('RGB') # Ensure the image only has RGB channels
        width, height = image.size

        # 2) Crop images to the center if they are above a certain size
        if width >= 400 and height >= 400:
          image = TF.center_crop(image, (400, 400))

        # 3) Apply transforms to the image (resize, tensor, normalize)
        if self.transform is not None:
            image = self.transform(image)

        # 4) Return the transformed image and index of label
        return (image, label_idx)

    # Get a list of class names
    def get_classes(self):
        return self.classes

print("Dataset Defined")


Dataset Defined


In [5]:
# Define the transformations to be applied to each image
train_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)), # Set standard image size
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2), 

    transforms.ToTensor(), # Normalizes values to a range of 0 - 1 instead of 0-255 for the RGB channels of pixels

    # Centers RGB values around zero using ImageNet standard mean/std. Helps training converge(the model learns and stabilizes) faster
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Kept the same the image structure for test images
test_transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create two separate datasets with different transforms (Only copies file list so two is fast still)
train_dataset = RoadDataset(root=base_path, transform=train_transform)
test_dataset = RoadDataset(root=base_path, transform=test_transform)

# Get class name array to assign based on indexes
class_names = train_dataset.get_classes()
print(class_names)

['dry', 'ice', 'mixed', 'snow']


In [6]:
# Decide split size for training (%90) and testing (%10)
train_size = int(0.9 * len(train_dataset))
test_size = len(train_dataset) - train_size

#====== Split up images randomly (using a seed for reproducibility) 
# Same seed ensures the same split but training gets the 90% with augmentation, testing gets the 10% without it
train_subset, _ = random_split(
    train_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

_, test_subset = random_split(
    test_dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

#=== Create training and testing dataloaders (DataLoaders wrap the datasets and handle feeding data to the model during training)
train_loader = DataLoader(
    train_subset,
    batch_size=32, # Amount of images to feed at a time (loss is averaged for each batch)
    shuffle=True, # Randomizes data order each run (epoch) so the model doesn't learn the patterns based on ordering
    num_workers=2, # How many next batches to load in the background with parallel threads while the model trains on the current batch
    pin_memory=True # Pre loads batches into GPU-ready memory which speeds up data loading. Only matters when using a GPU
)

test_loader = DataLoader(
    test_subset,
    batch_size=32,
    shuffle=False, # Set to false since order doesn't matter for testing
    num_workers=2,
    pin_memory=True
)

print("Data loaders setup")

Data loaders setup


In [7]:
# Neural network class that defines the model itself, inherits from PyTorch's base module (gives built-in training, saving, and GPU support)
class NeuralNet(nn.Module):

    # All neural network layers are defined here
    def __init__(self):
        # Initialize PyTorch's base module before defining our own layers (required for PyTorch to track weights)
        super().__init__()

        #======= Define layers ============#
        # Flow: Conv layers find patterns -> pooling simplifies -> FC layers decide what class it is.

        # Convolutional layer: slides a filter across the image to detect patterns (edges, textures, etc.)
        # Pool layer: shrinks the image by keeping the max value from each pool_size x pool_size block of pixels
        # Fully connected (FC) layer: combines detected patterns to make a final class prediction

        conv1_out = 12
        conv2_out = 24
        conv3_out = 48
        pool_size = 2

        fc1_size = 120
        num_classes = len(class_names)

        self.conv1 = nn.Conv2d(3, conv1_out, 5)        # 3 RGB channels -> 12 feature maps, 5x5 filter
        self.conv2 = nn.Conv2d(conv1_out, conv2_out, 5) # 12 -> 24 feature maps, 5x5 filter
        self.conv3 = nn.Conv2d(conv2_out, conv3_out, 3) # 24 -> 48 feature maps, 3x3 filter

        self.pool = nn.MaxPool2d(pool_size, pool_size) # Looks at every 2x2 block of pixels and keeps only the brightest one

        self.adapt_pool = nn.AdaptiveAvgPool2d((1, 1))  # Force feature maps down to 1x1 regardless of input size

        self.fc1 = nn.Linear(conv3_out, fc1_size) # Compress all pattern data into a smaller representation
        self.fc2 = nn.Linear(fc1_size, num_classes) # One output per road condition class, each is given a score and greatest one wins


    # Connects the layers in the order and way we want to run images through
    def forward(self, x):
        #== CONVOLUTIONAL
        x = self.pool(F.relu(self.conv1(x))) #F.relu sets any negative value to zero to increase class separation accuracy
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))

        #== ADAPTIVE POOL
        x = self.adapt_pool(x) # Shrink all feature maps to 1x1

        #== FLATTEN
        x = torch.flatten(x, 1)  # Flattens the maps of features into single rows of numbers

        #== NEURAL NETWORK
        x = F.relu(self.fc1(x))
        x = self.fc2(x) # No ReLU since negative scores mean "definitely not this category" and are useful
        return x

print("Neural Network defined")

Neural Network defined


In [8]:
# Set to True to load a previously trained model, False to train from scratch
load_saved_model = False

# Create an instance of the model
net = NeuralNet()
net.to(device) # To use GPU or CPU

if load_saved_model:
    net.load_state_dict(torch.load(save_path, map_location=device)) # Load the saved weights into the model (map_location ensures it works on GPU or CPU regardless of where it was trained)
    print("Loaded saved model from", save_path)
else:
    print("Starting with a fresh model")

# Used to handle loss calculations after the model predicts scores for each category
loss_function = nn.CrossEntropyLoss()

#==== Weighted loss to penalize mistakes on smaller classes more heavily
# Get total amount of files per catagory folder
class_counts_dict = Counter(label for _, label in train_dataset.samples)
print(class_counts_dict)

# Sort by label index so weights match class_names order (0=dry, 1=ice, 2=mixed, 3=snow)
class_counts = [class_counts_dict[i] for i in range(len(class_names))]
weights = torch.tensor([1.0 / c for c in class_counts])
weights = weights / weights.sum()
loss_function = nn.CrossEntropyLoss(weight=weights.to(device))
#======#

# Used to adjust the model's weights to reduce loss
optimizer = optim.SGD(
    net.parameters(), # Gives the optimizer access to all the weights in the model
    lr = 0.001, # sets the learning rate, so how big each adjustment step is. (Too big and the model overshoots, too small and it takes forever to learn)
    momentum = 0.9 # Used to keep 90% of the previous directions of modifications, and mix in 10% of the new direction
)

Starting with a fresh model
Counter({3: 5889, 0: 805, 1: 721, 2: 618})


In [9]:
# start the training, each epoch is an iteration of training where every image is seen once
for epoch in range(50):
    print(f'Training epoch {epoch+1}...')

    start = time.time() # Time each epoch

    # Used to see the average loss per epoch
    running_loss = 0

    # Run through the data in the defined batches
    for data in train_loader:
        # Unpack the images and labels of the batch, then load them on the avalible device
        images, labels = data
        images, labels = images.to(device), labels.to(device)

        # set all gradients (weight modifications from previous run) to zero
        optimizer.zero_grad()

        # Give the images to the model to classify and give scores. Runs the forward method in the NeuralNet class for each image
        outputs = net(images)

        # Calculate loss by comparing the predictions to the actual labels
        loss = loss_function(outputs, labels)

        # Figure out how much each weight contributed to the error. Each weight gets a gradient that says what direction and amount to adjust it for the next training tests
        loss.backward()

        # Apply the decided gradient to each weight, scaled by the learning rate defined in the optimizer
        optimizer.step()

        running_loss += loss.item() # Add up current loss for the epoch

    elapsed = time.time() - start
    # average loss accross all batches for the epoch, and total time
    print(f'Epoch {epoch+1} | Loss: {running_loss / len(train_loader):.4f} | Time: {elapsed:.1f}s\n')


Training epoch 1...
Epoch 1 | Loss: 1.3789 | Time: 289.7s

Training epoch 2...
Epoch 2 | Loss: 1.3268 | Time: 233.5s

Training epoch 3...
Epoch 3 | Loss: 1.2509 | Time: 229.1s

Training epoch 4...
Epoch 4 | Loss: 1.2222 | Time: 227.6s

Training epoch 5...
Epoch 5 | Loss: 1.1882 | Time: 229.7s

Training epoch 6...
Epoch 6 | Loss: 1.0732 | Time: 229.4s

Training epoch 7...
Epoch 7 | Loss: 0.9693 | Time: 229.4s

Training epoch 8...
Epoch 8 | Loss: 0.8250 | Time: 232.7s

Training epoch 9...
Epoch 9 | Loss: 0.7978 | Time: 226.3s

Training epoch 10...
Epoch 10 | Loss: 0.7549 | Time: 230.1s

Training epoch 11...
Epoch 11 | Loss: 0.7463 | Time: 230.7s

Training epoch 12...
Epoch 12 | Loss: 0.7404 | Time: 232.4s

Training epoch 13...
Epoch 13 | Loss: 0.7532 | Time: 229.5s

Training epoch 14...
Epoch 14 | Loss: 0.7237 | Time: 228.7s

Training epoch 15...
Epoch 15 | Loss: 0.7259 | Time: 229.3s

Training epoch 16...
Epoch 16 | Loss: 0.7205 | Time: 229.8s

Training epoch 17...
Epoch 17 | Loss: 0.71

In [10]:
# Export the weights/parameters, the model
torch.save(net.state_dict(), save_path)
print("Model saved")

Model saved


In [11]:
#====== Evaluate accuracy on any dataset (Call where needed) ======#
def evaluate(loader, dataName=""):
    correct = 0
    total = 0

    # Create a list the size of the classes to fill with amounts of correct predictionsin each class
    class_correct = [0] * len(class_names)
    class_total = [0] * len(class_names)

    # switch from training mode to evaluation mode, affects how some layers behave
    net.eval()

    # Run the test images through the model and determine the accuracy it has after training
    with torch.no_grad(): # Tells PyTorch not to track gradients during this section
        for data in loader:
            # Run the test data through the model
            images, labels = data
            images, labels = images.to(device), labels.to(device) # load them on the avalible device
            outputs = net(images)

            # Get the highest score of each image
            _, predicted = torch.max(outputs, 1)

            # Get the total amount of images and the amount that are correct where the predicted value matches the expected label
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Go through each image and determine the total of each class and how many images we predicted correctly
            for i in range(len(labels)):
                label_idx = labels[i].item()
                class_total[label_idx] += 1
                if predicted[i] == labels[i]:
                    class_correct[label_idx] += 1

    # For each class diplay the amount of correct predictions and the accuracy of each
    print(f"\n{dataName} Results:")
    print(f"Overall: {correct}/{total}  Accuracy: {100 * correct / total:.1f}%")
    for i, name in enumerate(class_names):
        acc = 100 * class_correct[i] / class_total[i] if class_total[i] > 0 else 0
        print(f"  {name}: {class_correct[i]}/{class_total[i]}  Accuracy: {acc:.1f}%")

print("Evaluation function loaded")

Evaluation function loaded


In [12]:
#====== Test on training test split(10%) ======#
evaluate(test_loader, "Test Set")


Test Set Results:
Overall: 605/804  Accuracy: 75.2%
  dry: 72/78  Accuracy: 92.3%
  ice: 75/75  Accuracy: 100.0%
  mixed: 45/61  Accuracy: 73.8%
  snow: 413/590  Accuracy: 70.0%


In [13]:
#====== Test on Cross Validation Dataset ======#
# Replace path as needed
cv_path = '/kaggle/input/datasets/johnson183/cv-images/cv_images'
cv_dataset = RoadDataset(root=cv_path, transform=test_transform)
cv_loader = DataLoader(cv_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"Testing on {len(cv_dataset)} external images")
evaluate(cv_loader, "External CV Images")

Testing on 60 external images

External CV Images Results:
Overall: 20/60  Accuracy: 33.3%
  dry: 6/15  Accuracy: 40.0%
  ice: 1/15  Accuracy: 6.7%
  mixed: 5/15  Accuracy: 33.3%
  snow: 8/15  Accuracy: 53.3%


In [14]:
#==== Test one image with the model =====#
# Enure you run these cells first:
# 1. Imports
# 2. Path setup and constants
# 3. The RoadDataset class and transforms (needed for post_crop_transform and class_names)
# 4. The NeuralNet class definition
# 5. Model creation with load_saved_model = True (loads your saved weights)

# def predict_image(image_path):
#     image = Image.open(image_path)
#     width, height = image.size

#     if width >= 400 and height >= 400:
#         image = TF.center_crop(image, (400, 400))

#     image = image.convert('RGB') # Ensure the image only has RGB channels
#     image = post_crop_transform(image)
#     image = image.unsqueeze(0).to(device)  # To allow a single image to run since it expects a batch

#     net.eval()
#     with torch.no_grad():
#         outputs = net(image)
#         _, predicted = torch.max(outputs, 1)

#     print(f"Predicted: {class_names[predicted.item()]}")

# predict_image('/image/path') # Replace with path of image to test